In [6]:
import pandas as pd
file_path = '/Users/callumhall/Documents/Dissertation/Datasets/Archive/NHS Data/Annual Prescriptions Summary.csv'
df_prescriptions = pd.read_csv(file_path)

In [8]:
df_prescriptions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1496 entries, 0 to 1495
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   PRACTICE_CODE   1496 non-null   object
 1   POSTCODE        1496 non-null   object
 2   Total_MH_Items  1496 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 35.2+ KB


In [10]:
import pandas as pd

# =============================================================================
# PHASE 1: SETUP AND LOADING
# =============================================================================
paths = {
    'mh_prescriptions': '/Users/callumhall/Documents/Dissertation/Datasets/Cleaned Data/Annual Prescriptions Summary.csv',
    'master_lookup': '/Users/callumhall/Documents/Dissertation/Datasets/master_lookup_2021_based.csv'
}
df_mh_practice = pd.read_csv(paths['mh_prescriptions'])
df_master_lookup = pd.read_csv(paths['master_lookup'])
print("Files loaded successfully.")

# =============================================================================
# PHASE 2: STANDARDIZE POSTCODES AND AGGREGATE
# =============================================================================
print("\nStandardizing postcode formats...")
df_mh_practice['POSTCODE_CLEAN'] = df_mh_practice['POSTCODE'].astype(str).str.upper().str.replace(' ', '')
df_master_lookup['pcds_CLEAN'] = df_master_lookup['pcds'].astype(str).str.upper().str.replace(' ', '')

df_mh_with_lsoa = pd.merge(
    df_mh_practice, 
    df_master_lookup[['pcds_CLEAN', 'LSOA21CD']], 
    left_on='POSTCODE_CLEAN', 
    right_on='pcds_CLEAN', 
    how='left'
)
df_mh_agg = df_mh_with_lsoa.groupby('LSOA21CD')['Total_MH_Items'].sum().reset_index()
df_mh_agg.rename(columns={'LSOA21CD': 'LSOA_Code'}, inplace=True)
print("Prescription data aggregated to LSOA level.")

# =============================================================================
# PHASE 3: CHECK THE NUMBER OF ZEROS
# =============================================================================
zero_count = df_mh_agg[df_mh_agg['Total_MH_Items'] == 0].shape[0]
total_lsoas_with_prescriptions = len(df_mh_agg)

print(f"\n--- Final Check ---")
print(f"Number of LSOAs with a raw prescription count of 0: {zero_count}")
print(f"Total number of LSOAs with at least one GP practice: {total_lsoas_with_prescriptions}")

Files loaded successfully.

Standardizing postcode formats...
Prescription data aggregated to LSOA level.

--- Final Check ---
Number of LSOAs with a raw prescription count of 0: 0
Total number of LSOAs with at least one GP practice: 1039
